In [ ]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")

In [ ]:
sys.path.append("../../../")

In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, sparse_matrices, benchmark, field_sampler, mesh_utilities
import inflatables_parametrization as parametrization, numpy as np, importlib, pickle, wall_generation
import utils
import py_newton_optimizer
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf

target_surf = mesh.Mesh("../../../SiggraphExamples/Meshes/20200106_teaser_srf_v3_remeshed.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=True))
target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
# Choose reasonable stretching bounds in terms of the relative fusing curve widths.
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(2, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
print(alphaMin, alphaMax)

In [ ]:
# Run some iterations of the local-global algorithm to ensure a good separation between singular values.
# This step can also be used as a prediction of the feasiblity of a design surface:
# if it is unable to nearly satisfy the singular value constraints,
# the surface is probably infeasible.
lg = parametrization.LocalGlobalParametrizer(target_surf, parametrization.lscm(target_surf))

lg.alphaMin = 1.4
lg.alphaMax = np.pi / 2
print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
visualization.visualize(lg)

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(target_surf, lg.uv())
rparam.alphaMin = alphaMin
rparam.alphaMax = alphaMax

In [ ]:
def optimize_rparam(param, alphaRegW, phiRegW, bendRegW):
    param.alphaRegW = alphaRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    opts = NewtonOptimizerOptions()
    opts.niter = 2000
    opts.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
    #opts.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()
    cr = parametrization.regularized_parametrization_newton(param, param.rigidMotionPinVars, opts)

In [ ]:
# Rerunning this cell a couple times can improve the results
benchmark.reset()
with suppress_stdout(): optimize_rparam(rparam, 100.0, 10.0, 500.0)
with suppress_stdout(): optimize_rparam(rparam, 10.0, 1.0, 250.0)
with suppress_stdout(): optimize_rparam(rparam, 1.0, 0.1, 125.0)
with suppress_stdout(): optimize_rparam(rparam, 0.1, 0.01, 62.5)
with suppress_stdout(): optimize_rparam(rparam, 0.1, 0.01, 31.25)
benchmark.report()

In [ ]:
importlib.reload(utils)

In [ ]:
# Report the values and gradients of each objective term
print(f'Energies: {utils.allEnergies(rparam)}')
print(f'Gradient Norms: {utils.allGradientNorms(rparam)}')

In [ ]:
# Visualize the flattening
visualization.visualize(rparam)

In [ ]:
importlib.reload(visualization)

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False, width = 10, height = 10)

## Upsampling and channel generation

In [ ]:
nsubdiv=4
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)
upsampledStretches = np.clip(upsampledStretches, alphaMin, alphaMax)
(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_stripe_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles,
                                                                    wwf.canonicalWallWidthForStretchFactor(upsampledStretches), frequency=1)

In [ ]:
import pickle, mesh, wall_generation, visualization, numpy as np

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, height=12)

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=4.0,
                                              minContourLen=10)

In [ ]:
visualization.plot_line_segments(pts, edges, width=10, height=16)

## Meshing and inflation simulation

In [ ]:
m, fuseMarkers, edgeMarkers = wall_generation.triangulate_channel_walls(pts[:,0:2], edges, triArea=1.0)
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=20, height=18)

In [ ]:
import sheet_meshing, inflation

In [ ]:
import importlib
importlib.reload(sheet_meshing)

In [ ]:
m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, pts, edges, triArea=1)

In [ ]:
isheet = inflation.InflatableSheet(m, iwv)
isheet.setRelaxedStiffnessEpsilon(1e-6)
uv = rparam.uv()

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
# Manually stretch the sheet onto the target surface by applying the inverse of the parametrization
paramSampler = field_sampler.FieldSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-4
opts.niter = 1000

In [ ]:
# Fix the boundary positions
import boundaries
bdryVars = boundaries.getBoundaryVars(isheet)
# fixedVars = bdryVars

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
import boundaries
bdryVars = boundaries.getOuterBoundaryVars(isheet)
fixedVars = bdryVars

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [ ]:
isheet.pressure = 1e-5

In [ ]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [ ]:
isheet.pressure = 0.03

In [ ]:
opts.niter = 1000

In [ ]:
import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);